In [85]:
import pandas as pd
import numpy as np
import warnings

In [139]:
# Seteamos las rutas
bd_2020 = {
    'Enero': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/ENERO.xlsx',
    'Febrero': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/FEBRERO.xlsx',
    'Marzo': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/MARZO.xlsx',
    'Abril': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/ABRIL.xlsx',
    'Mayo': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/MAYO.xlsx',
    'Junio': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/JUNIO.xlsx',
    'Julio': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/JULIO.xlsx',
    'Agosto': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/AGOSTO.xlsx',
    'Setiembre': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/SETIEMBRE.xlsx',
    'Octubre': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/OCTUBRE.xlsx',
    'Noviembre': 'C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/NOVIEMBRE.xlsx'
}

In [140]:
def cargar_c18_limpio(ruta, hoja='c-18'):
    
    df_temp = pd.read_excel(ruta, sheet_name=hoja, header=None)
    
    fila_header = None
    
    for i, row in df_temp.iterrows():
        if row.astype(str).str.contains('Años', case=False).any():
            fila_header = i
            break
    
    if fila_header is None:
        raise ValueError(f"No se encontró header real en {ruta}")
    
    # ahora sí leemos bien
    df = pd.read_excel(
        ruta,
        sheet_name=hoja,
        skiprows=fila_header
    )
    
    # limpiar columnas
    df.columns = df.columns.str.strip()
    
    # eliminar filas vacías
    df = df.dropna(how='all').reset_index(drop=True)
    
    return df

In [141]:
def limpiar_basico(df):
    
    # rellenar región hacia abajo
    df['Región'] = df['Región'].ffill()
    
    # eliminar total nacional (no lo quieres)
    df = df[df['Región'] != 'Total Nacional']
    
    # limpiar año
    df['Años'] = (
        df['Años']
        .astype(str)
        .str.extract(r'(\d{4})')[0]
        .astype(float)
    )
    
    # eliminar filas sin año
    df = df.dropna(subset=['Años'])
    
    return df

In [142]:
warnings.filterwarnings("ignore")
dfs = []

for mes, ruta in bd_2020.items():
    
    df = cargar_c18_limpio(ruta)
    df = limpiar_basico(df)
    
    df['Mes'] = mes
    
    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

In [143]:
# Corregimos nombres de regiones
df_final['Región'] = (
    df_final['Región']
    .str.normalize('NFKD')
    .str.encode('ascii', errors='ignore')
    .str.decode('utf-8')
)

# Filtramos solo ciertos departamentos
departamentos = ['Piura', 'La Libertad', 'Ica', 'San Martin', 'Junin', 'Puno']
df_final = df_final[df_final['Región'].isin(departamentos)]

# Corregimos nombres de los meses
df_final['Mes'] = df_final['Mes'].replace({
    'Setiembre': 'Septiembre'
})

In [144]:
# Ahora sí ordenamos los meses
orden_meses = ['Enero','Febrero','Marzo','Abril','Mayo','Junio',
               'Julio','Agosto','Septiembre','Octubre','Noviembre','Diciembre']

df_final['Mes'] = pd.Categorical(df_final['Mes'], categories=orden_meses, ordered=True)

df_final = df_final.sort_values(['Región','Años','Mes'])

In [145]:
# Limpiamos columnas
df_final.columns = (
    df_final.columns
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

# Eliminamos duplicados
df_final = df_final.loc[:, ~df_final.columns.duplicated()]

# Nos quedamos solo con los cultivos
cols_excluir = ['región', 'años', 'mes']
cols_cultivos = [col for col in df_final.columns if col not in cols_excluir]

# Convertimos a numérico esas columnas
for col in cols_cultivos:
    df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

In [146]:
# Ahora hacemos la agrupación por años para tener el acumulado
for col in cols_cultivos:
    df_final[col] = (
        df_final
        .groupby(['región','años'])[col]
        .ffill()
    )

# Ahora hacemos la agrupación por años para tener el mensual
for col in cols_cultivos:
    df_final[f'{col}_mensual'] = (
        df_final
        .groupby(['región','años'])[col]
        .diff()
        .fillna(df_final[col])
    )

In [147]:
print(df_final.dtypes.head(20))

región            object
años             float64
trigo            float64
maíz a. duro     float64
maíz amiláceo    float64
arroz cáscara    float64
cebada grano     float64
quinua           float64
espárrago        float64
alcachofa        float64
ají              float64
piquillo         float64
pimiento         float64
región.1         float64
años.1           float64
tomate           float64
zapallo          float64
arveja verde     float64
zanahoria        float64
ajo              float64
dtype: object


In [148]:
print(df_final.head())

    región    años  trigo  maíz a. duro  maíz amiláceo  arroz cáscara  \
20     Ica  2019.0    0.0     11588.260            0.0            0.0   
72     Ica  2019.0    0.0     24559.130           15.7            0.0   
124    Ica  2019.0    0.0     41119.319           15.7            0.0   
176    Ica  2019.0    0.0     51307.969           15.7            0.0   
228    Ica  2019.0    0.0     51307.969           15.7            0.0   

     cebada grano  quinua  espárrago  alcachofa  ...  algodón rama_mensual  \
20            0.0    23.0   19220.45        0.0  ...              1837.797   
72            0.0    23.0   31818.55        0.0  ...              2528.450   
124           0.0    23.0   48566.65        0.0  ...              3225.810   
176           0.0    23.0   61106.69        0.0  ...              7646.300   
228           0.0    23.0   73440.36        0.0  ...             10431.180   

     orégano_mensual  arándano_mensual  espá- rrago_mensual  \
20               0.0         

In [149]:
df_final.to_csv('C:/Users/Joyssie/Documents/Data Minning/Proyecto/BDS/2020/BD_FINAL.csv', index=False)